# Open-Source LLM Strategy

## Completed Practical Exercises

This notebook completes the six exercises on:

- levels of model openness;
- license compatibility for SaaS;
- model selection under hardware and language constraints;
- local deployment readiness;
- benchmark-driven model matching;
- local versus cloud deployment planning.

The analysis distinguishes **open source**, **open weights**, and
**source-available models**. These terms are not interchangeable.

## How to use this notebook

- Run the code cells from top to bottom.
- Exercise 4 audits the current runtime. In Google Colab, it audits the
  Colab machine, not your personal computer.
- The optional 7B inference cell is disabled by default to avoid a large
  download.
- License notes are educational summaries, not legal advice. Before
  production use, read the complete license and acceptable-use policy.

In [ ]:
%pip install -q "pandas>=2.0,<3.0" "psutil>=5.9,<7.0"

In [ ]:
import os
import platform
import shutil
import subprocess
from pathlib import Path
from typing import Dict, List

import pandas as pd
import psutil
from IPython.display import display

pd.set_option("display.max_colwidth", 140)

print("Runtime:", platform.platform())
print("Python:", platform.python_version())

# Exercise 1 — Open-Source Levels Reflection

## 1.1 Definitions and key characteristics

In [ ]:
open_source_levels = {
    "Fully Open": {
        "what_is_open": (
            "Model architecture, pretrained weights, training and inference "
            "code, evaluation code, documentation, and sufficient training-"
            "data information or provenance under licenses that permit use, "
            "study, modification, and redistribution."
        ),
        "what_it_enables": (
            "You can inspect the full pipeline, reproduce or adapt it, "
            "fine-tune it, audit it, and redistribute compliant derivatives."
        ),
        "main_limitation": (
            "Full reproducibility can still be expensive, and openness does "
            "not automatically guarantee privacy, safety, data quality, or "
            "regulatory compliance."
        ),
    },
    "Weights Released": {
        "what_is_open": (
            "Pretrained weights are downloadable, usually with inference "
            "code and a model card, but training data, complete training code, "
            "or unrestricted licensing may be missing."
        ),
        "what_it_enables": (
            "You can run and often fine-tune or quantize the model if the "
            "license permits it, but you may not be able to reproduce, fully "
            "audit, or freely redistribute the original training process."
        ),
        "main_limitation": (
            "The license may impose use restrictions, attribution, user-scale "
            "thresholds, or acceptable-use conditions."
        ),
    },
    "Architecture Only": {
        "what_is_open": (
            "The paper, architecture description, or implementation is "
            "available, but pretrained weights are not released."
        ),
        "what_it_enables": (
            "You can inspect and reimplement the structure, but must train "
            "the model from scratch before it has useful pretrained capability."
        ),
        "main_limitation": (
            "Training from scratch requires substantial data, compute, "
            "engineering, and evaluation resources."
        ),
    },
}

openness_df = pd.DataFrame(open_source_levels).T.reset_index()
openness_df = openness_df.rename(columns={"index": "openness_level"})
display(openness_df)

## 1.2 Side-by-side comparison

| Openness level | What is open? | Impact on retraining or modification |
|---|---|---|
| Fully Open | Architecture, weights, core code, documentation, and sufficient training information | Full inspection, adaptation, retraining, auditing, and compliant redistribution are possible |
| Weights Released | Pretrained parameters, often inference code | Inference and fine-tuning may be possible, but full reproduction and redistribution depend on missing assets and license terms |
| Architecture Only | Design or source implementation | The structure can be studied and reimplemented, but useful pretrained behavior requires training from scratch |

## 1.3 Comparative paragraph

A fully open model exposes the main components needed to study, modify,
retrain, and redistribute the system, making it the strongest option for
reproducibility and independent auditing. A weights-released model provides
useful pretrained parameters and may support fine-tuning, but missing
training data, code, or restrictive terms can limit transparency and
commercial freedom. An architecture-only release explains how the model is
built but provides no pretrained capability, so the user must fund training
from scratch. Openness is therefore a spectrum, and the practical rights
always depend on the attached licenses.

## 1.4 Healthcare prompt answer

For a healthcare assistant that must be retrained and audited on sensitive
clinical data, **Fully Open** is the preferred level because it offers the
greatest control over weights, training code, evaluation, and deployment.
Open weights may technically permit fine-tuning, but high-stakes healthcare
also requires a compatible license, data governance, privacy controls,
clinical validation, security testing, and human oversight.

# Exercise 2 — License Check for SaaS Use

## Selected models

1. `mistralai/Mistral-7B-Instruct-v0.3`  
   https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3

2. `meta-llama/Llama-2-7b-chat-hf`  
   https://huggingface.co/meta-llama/Llama-2-7b-chat-hf

## 2.1 Completed Markdown checklist

### Mistral-7B-Instruct-v0.3

- [x] **Model:** `mistralai/Mistral-7B-Instruct-v0.3`
- [x] **License name:** Apache License 2.0
- [x] **Commercial use:** Yes
- [x] **Modification and redistribution:** Allowed under Apache 2.0
- [x] **Attribution and notices:** Preserve required copyright, license,
  attribution, and NOTICE information; state significant modifications
- [x] **Monthly active-user limit:** No model-specific MAU threshold found
  in the Apache 2.0 model license
- [x] **Safety note:** The model card states that the model has no built-in
  moderation mechanisms
- [x] **Export and legal compliance:** The deployer remains responsible for
  applicable laws and regulations

### Llama-2-7b-chat-hf

- [x] **Model:** `meta-llama/Llama-2-7b-chat-hf`
- [x] **License name:** Llama 2 Community License Agreement
- [x] **Commercial use:** Conditional
- [x] **Access:** Hugging Face access is gated and requires acceptance of
  Meta's terms
- [x] **Redistribution:** Include a copy of the agreement and retain the
  required attribution notice
- [x] **Large-platform threshold:** An organization with more than
  700 million monthly active users at the Llama 2 release date must request
  a separate license from Meta
- [x] **Use restriction:** Llama materials or outputs may not be used to
  improve another large language model, except Llama 2 derivatives
- [x] **Acceptable-use and trade compliance:** Use must comply with the
  Acceptable Use Policy, laws, and trade regulations
- [x] **Open-source classification:** More accurately described as
  open-weight or source-available than fully open source because the custom
  license imposes field-of-use and scale restrictions

In [ ]:
license_checklist = pd.DataFrame([
    {
        "model": "mistralai/Mistral-7B-Instruct-v0.3",
        "license": "Apache-2.0",
        "commercial_use": "Allowed",
        "attribution_or_notice": (
            "Preserve license/copyright/NOTICE requirements and mark changes"
        ),
        "special_restrictions": (
            "No model-specific MAU cap; deployer handles legal compliance"
        ),
        "saas_assessment": (
            "Generally SaaS-friendly, subject to Apache-2.0 compliance "
            "and product-level safety obligations"
        ),
    },
    {
        "model": "meta-llama/Llama-2-7b-chat-hf",
        "license": "Llama 2 Community License",
        "commercial_use": "Conditional",
        "attribution_or_notice": (
            "Retain attribution and include the agreement on redistribution"
        ),
        "special_restrictions": (
            ">700M MAU separate license; acceptable-use terms; restrictions "
            "on improving non-Llama LLMs"
        ),
        "saas_assessment": (
            "Possible for many SaaS products, but requires a detailed "
            "license and acceptable-use review"
        ),
    },
])

display(license_checklist)

## 2.2 SaaS compatibility decision checklist

Before choosing any model for a commercial product, verify:

- [x] The exact model repository and version
- [x] The exact license attached to weights, code, tokenizer, and dataset
- [x] Whether commercial use is permitted
- [x] Whether derivative models may be redistributed
- [x] Attribution, NOTICE, and naming requirements
- [x] User-count, revenue, geographic, or field-of-use restrictions
- [x] Acceptable-use and prohibited-use policies
- [x] Export-control and sanctions obligations
- [x] Training-data and intellectual-property risk
- [x] Privacy, security, and safety obligations for the product
- [x] A process for monitoring future license or model-card changes

# Exercise 3 — LLM Matchmaker Challenge

## 3.1 Search-filter summary

In [ ]:
filters_by_team = {
    "LegalTech": {
        "constraints": [
            "CPU or quantized inference",
            "7B parameters or fewer where possible",
            "strong logic and instruction following",
            "commercially usable license preferred",
        ],
        "hugging_face_search": (
            "https://huggingface.co/models?pipeline_tag=text-generation"
            "&sort=trending&search=instruct"
        ),
        "evaluation_focus": [
            "MMLU", "BoolQ", "HellaSwag", "latency", "RAM", "license"
        ],
    },
    "EdTech": {
        "constraints": [
            "math and logic specialization",
            "low-end laptop footprint",
            "quantized CPU deployment",
        ],
        "hugging_face_search": (
            "https://huggingface.co/models?pipeline_tag=text-generation"
            "&sort=trending&search=math"
        ),
        "evaluation_focus": [
            "GSM8K", "MATH", "model size", "tool use", "language coverage"
        ],
    },
    "Global NGO": {
        "constraints": [
            "at least five languages",
            "moderate memory footprint",
            "quantized deployment preferred",
        ],
        "hugging_face_search": (
            "https://huggingface.co/models?pipeline_tag=text-generation"
            "&sort=trending&search=multilingual"
        ),
        "evaluation_focus": [
            "language count", "FLORES or multilingual evaluations",
            "license", "regional-language testing"
        ],
    },
}

for team, details in filters_by_team.items():
    print("\n", team)
    for key, value in details.items():
        print(f"- {key}: {value}")

## 3.2 Candidate lists

Scores from different model cards are not automatically comparable because
prompts, shot counts, evaluation code, and model versions may differ.
The table therefore labels the source or caveat for each reported result.

In [ ]:
candidates_by_team = {
    "LegalTech": [
        {
            "model": "microsoft/Phi-3.5-mini-instruct",
            "parameters": "3.8B",
            "license": "MIT",
            "architecture_or_strength": (
                "Compact decoder model; designed for constrained and "
                "low-latency environments"
            ),
            "quantization": "GGUF/int4 community conversions available",
            "reported_evidence": (
                "MMLU 69.0, BoolQ 78.0, HellaSwag 69.4 "
                "in the model-card comparison protocol"
            ),
        },
        {
            "model": "mistralai/Mistral-7B-Instruct-v0.3",
            "parameters": "7B",
            "license": "Apache-2.0",
            "architecture_or_strength": (
                "General instruction model with function-calling support"
            ),
            "quantization": "GGUF, 4-bit and 8-bit variants available",
            "reported_evidence": (
                "MMLU 60.3, BoolQ 80.5, HellaSwag 71.6 "
                "in the same Phi-3.5 comparison table"
            ),
        },
        {
            "model": "Qwen/Qwen2.5-3B-Instruct",
            "parameters": "3.09B",
            "license": "Qwen Research License",
            "architecture_or_strength": (
                "Compact instruction model with long context and "
                "multilingual support"
            ),
            "quantization": "Official/community AWQ, GPTQ and GGUF variants",
            "reported_evidence": (
                "Model card states support for 29+ languages; perform a "
                "domain-specific legal benchmark before selection"
            ),
        },
    ],
    "EdTech": [
        {
            "model": "Qwen/Qwen2.5-Math-1.5B-Instruct",
            "parameters": "1.5B family (~2B displayed on HF)",
            "license": "Apache-2.0",
            "architecture_or_strength": (
                "Math-specialized instruction model for English and Chinese"
            ),
            "quantization": "Small enough for practical 4-bit CPU testing",
            "reported_evidence": (
                "MATH with tool-integrated reasoning reported at 79.7; "
                "the score depends on the TIR setup"
            ),
        },
        {
            "model": "microsoft/Phi-3.5-mini-instruct",
            "parameters": "3.8B",
            "license": "MIT",
            "architecture_or_strength": (
                "Strong small-model reasoning and broad general utility"
            ),
            "quantization": "GGUF/int4 community conversions available",
            "reported_evidence": (
                "GSM8K 86.2 and MATH 48.5 in the model-card protocol"
            ),
        },
        {
            "model": "deepseek-ai/deepseek-math-7b-instruct",
            "parameters": "7B",
            "license": "DeepSeek Model License",
            "architecture_or_strength": (
                "Math-specialized model with chain-of-thought style capability"
            ),
            "quantization": "4-bit community variants available",
            "reported_evidence": (
                "Commercial use is supported by its model license; "
                "larger footprint than the other candidates"
            ),
        },
    ],
    "Global NGO": [
        {
            "model": "bigscience/bloomz-3b",
            "parameters": "3B",
            "license": "BigScience BLOOM RAIL 1.0",
            "architecture_or_strength": (
                "Instruction-tuned multilingual BLOOM family"
            ),
            "quantization": "8-bit/4-bit loading and community GGUF options",
            "reported_evidence": (
                "Model card documents 46 natural languages and reports "
                "multilingual XWinograd results"
            ),
        },
        {
            "model": "Qwen/Qwen2.5-3B-Instruct",
            "parameters": "3.09B",
            "license": "Qwen Research License",
            "architecture_or_strength": (
                "More recent compact multilingual instruction model"
            ),
            "quantization": "AWQ, GPTQ and GGUF variants available",
            "reported_evidence": "Model card states support for 29+ languages",
        },
        {
            "model": "Qwen/Qwen2.5-7B-Instruct",
            "parameters": "7.61B actual",
            "license": "Apache-2.0",
            "architecture_or_strength": (
                "Quality-first multilingual alternative with long context"
            ),
            "quantization": "AWQ, GPTQ and GGUF variants available",
            "reported_evidence": (
                "29+ languages; exceeds the strict 7B threshold slightly"
            ),
        },
    ],
}

candidate_rows = []
for team, candidates in candidates_by_team.items():
    for candidate in candidates:
        candidate_rows.append({"team": team, **candidate})

display(pd.DataFrame(candidate_rows))

## 3.3 Best-fit comparison and final picks

In [ ]:
matchmaker_table = pd.DataFrame([
    {
        "team": "LegalTech",
        "needs": "Fast, logic-heavy chatbot on CPU",
        "your_pick": "microsoft/Phi-3.5-mini-instruct",
        "why": (
            "3.8B footprint, MIT license, strong MMLU/GSM8K results, and "
            "explicit positioning for constrained low-latency environments"
        ),
        "important_caveat": (
            "Use legal-document RAG, citations, domain tests, access control, "
            "and mandatory human review; benchmark strength is not legal accuracy"
        ),
    },
    {
        "team": "EdTech",
        "needs": "Math and logic on low-end laptops",
        "your_pick": "Qwen/Qwen2.5-Math-1.5B-Instruct",
        "why": (
            "Smallest math-specialized candidate, Apache-2.0, and strong "
            "reported MATH performance when tool-integrated reasoning is used"
        ),
        "important_caveat": (
            "Mainly English and Chinese; validate without tools and on the "
            "actual curriculum before deployment"
        ),
    },
    {
        "team": "Global NGO",
        "needs": "Five or more languages with a modest footprint",
        "your_pick": "bigscience/bloomz-3b",
        "why": (
            "3B footprint and documented coverage of 46 natural languages"
        ),
        "important_caveat": (
            "It is older and may underperform recent models; test every target "
            "language, dialect, cultural context, and safety category"
        ),
    },
])

display(matchmaker_table)

### Selection conclusion

- **LegalTech:** Phi-3.5-mini is the best starting point for constrained
  hardware, but no general benchmark substitutes for legal-domain evaluation.
- **EdTech:** Qwen2.5-Math-1.5B is the best footprint-to-specialization match.
- **Global NGO:** BLOOMZ-3B best satisfies the strict multilingual and size
  constraints; Qwen2.5-7B-Instruct is a quality-first alternative if a
  slightly larger model is acceptable.

# Exercise 4 — Local Readiness Audit

The following cell audits the machine running this notebook. In Colab, it
reports the Colab VM. To audit your own machine, run the notebook locally or
enter your personal specifications in the override cell.

In [ ]:
def read_cpu_flags() -> List[str]:
    flags = []

    if Path("/proc/cpuinfo").exists():
        text = Path("/proc/cpuinfo").read_text(
            encoding="utf-8",
            errors="ignore",
        ).lower()
        for flag in ["avx512", "avx2", "avx", "sse4_2", "sse4_1", "neon"]:
            if flag in text:
                flags.append(flag.upper())

    if platform.system() == "Darwin":
        try:
            output = subprocess.check_output(
                ["sysctl", "-a"],
                text=True,
                stderr=subprocess.DEVNULL,
            ).lower()
            for flag in ["avx2", "avx1.0", "sse4.2", "neon"]:
                if flag in output:
                    flags.append(flag.upper())
        except Exception:
            pass

    return sorted(set(flags))


total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
free_disk_gb = shutil.disk_usage("/").free / (1024 ** 3)
system_name = platform.system()
release = platform.release()
os_description = f"{system_name} {release}"
is_wsl = "microsoft" in release.lower() or "wsl" in release.lower()
course_os_match = system_name == "Linux" or is_wsl
modern_llama_cpp_os_match = system_name in {"Linux", "Darwin", "Windows"}

system_specs = {
    "ram_gb": round(total_ram_gb, 2),
    "free_disk_gb": round(free_disk_gb, 2),
    "os": os_description,
    "machine_architecture": platform.machine(),
    "cpu_flags_detected": read_cpu_flags(),
    "gcc_or_clang": (
        shutil.which("gcc")
        or shutil.which("clang")
        or shutil.which("cc")
    ),
    "cmake": shutil.which("cmake"),
    "make": shutil.which("make"),
}

system_specs

In [ ]:
readiness_table = pd.DataFrame([
    {
        "requirement": "RAM (>= 16 GB)",
        "current_runtime": f"{system_specs['ram_gb']} GB",
        "meets_course_requirement": system_specs["ram_gb"] >= 16,
    },
    {
        "requirement": "Free disk space (>= 40 GB)",
        "current_runtime": f"{system_specs['free_disk_gb']} GB",
        "meets_course_requirement": system_specs["free_disk_gb"] >= 40,
    },
    {
        "requirement": "OS (Linux or WSL2 in course rubric)",
        "current_runtime": system_specs["os"],
        "meets_course_requirement": course_os_match,
    },
])

readiness_table["status"] = readiness_table[
    "meets_course_requirement"
].map({True: "✅", False: "❌"})

display(readiness_table)

## Personal-machine override

Replace the values below with your own computer specifications if the current
runtime is Colab. Keep `None` only when the value is unknown.

In [ ]:
personal_system_specs = {
    "ram_gb": None,
    "free_disk_gb": None,
    "os": None,
    "cpu_architecture": None,
}

print(
    "Enter your personal values above when you want the report to describe "
    "your own computer rather than the notebook runtime."
)

## 4.1 llama.cpp readiness

Modern `llama.cpp` supports:

- x86 processors with optimizations such as AVX, AVX2, AVX512, and AMX;
- ARM NEON;
- Apple Silicon through Accelerate and Metal;
- GGUF models and low-bit quantization;
- CPU-only and CPU/GPU hybrid inference.

The course rubric mentions Linux or WSL2, but current `llama.cpp` also
supports macOS and Windows. Building from source generally requires CMake
and a C/C++ compiler; prebuilt packages and binaries can avoid a local build.

In [ ]:
llama_cpp_readiness = {
    "os_supported_by_modern_llama_cpp": modern_llama_cpp_os_match,
    "course_linux_or_wsl2_requirement": course_os_match,
    "architecture": system_specs["machine_architecture"],
    "cpu_flags_detected": system_specs["cpu_flags_detected"],
    "compiler_detected": bool(system_specs["gcc_or_clang"]),
    "compiler_path": system_specs["gcc_or_clang"],
    "cmake_detected": bool(system_specs["cmake"]),
    "make_detected": bool(system_specs["make"]),
    "recommended_model_format": "GGUF",
    "recommended_7b_quantization": (
        "Start with Q4_K_M; use Q5_K_M if RAM permits and quality matters"
    ),
}

display(pd.DataFrame(
    llama_cpp_readiness.items(),
    columns=["check", "result"],
))

## 4.2 Upgrade actions

- **RAM below 16 GB:** Prefer a 1B–4B GGUF model, close other applications,
  or use cloud inference. Swap may prevent crashes but is much slower than RAM.
- **Disk below 40 GB:** Remove unused models, caches, containers, or virtual
  machines; use external storage for model files.
- **Unsupported or inconvenient OS:** Use native macOS support, Linux, WSL2,
  a prebuilt binary, Docker, or a cloud notebook.
- **Missing compiler/CMake:** Install Xcode Command Line Tools on macOS,
  build-essential and CMake on Linux, or Visual Studio Build Tools on Windows.
- **CPU lacks modern vector instructions:** Choose smaller quantized models
  or use a cloud GPU; inference may still work but can be slow.

In [ ]:
upgrade_actions = []

if system_specs["ram_gb"] < 16:
    upgrade_actions.append(
        "Use a <=4B quantized model, add RAM, or move inference to cloud."
    )

if system_specs["free_disk_gb"] < 40:
    upgrade_actions.append(
        "Free disk space or attach external storage before keeping several models."
    )

if not modern_llama_cpp_os_match:
    upgrade_actions.append(
        "Use Linux/WSL2 or another supported llama.cpp environment."
    )

if not system_specs["cmake"] or not system_specs["gcc_or_clang"]:
    upgrade_actions.append(
        "Install CMake and a supported C/C++ compiler, or use prebuilt binaries."
    )

if not upgrade_actions:
    upgrade_actions.append(
        "The detected runtime meets the basic resource checks; run a small "
        "GGUF benchmark before committing to a 7B model."
    )

for action in upgrade_actions:
    print("-", action)

# Exercise 5 — Benchmark-Based Model Explorer

## Methodological note

The current Open LLM Leaderboard has evolved and its active metric set is
not identical to the historical HellaSwag/MMLU exercise. To keep the scores
comparable, the table below uses three values published under one comparative
protocol in the Phi-3.5 model card. Do not mix scores from unrelated harness
versions, prompts, shot counts, or model variants without labeling them.

In [ ]:
leaderboard_models = [
    "microsoft/Phi-3.5-mini-instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "google/gemma-2-9b-it",
]

benchmark_table = pd.DataFrame([
    {
        "model_name": "microsoft/Phi-3.5-mini-instruct",
        "model_url": (
            "https://huggingface.co/microsoft/Phi-3.5-mini-instruct"
        ),
        "hellaswag_score": 69.4,
        "mmlu_score": 69.0,
        "license_type": "MIT",
        "ideal_use_case": (
            "Compact academic/reasoning assistant for constrained hardware"
        ),
        "protocol_note": (
            "Phi-3.5 model-card comparison; HellaSwag 5-shot, MMLU 5-shot"
        ),
    },
    {
        "model_name": "mistralai/Mistral-7B-Instruct-v0.3",
        "model_url": (
            "https://huggingface.co/mistralai/"
            "Mistral-7B-Instruct-v0.3"
        ),
        "hellaswag_score": 71.6,
        "mmlu_score": 60.3,
        "license_type": "Apache-2.0",
        "ideal_use_case": (
            "Commercial general assistant or support agent with domain RAG"
        ),
        "protocol_note": (
            "Same Phi-3.5 comparative protocol"
        ),
    },
    {
        "model_name": "google/gemma-2-9b-it",
        "model_url": (
            "https://huggingface.co/google/gemma-2-9b-it"
        ),
        "hellaswag_score": 80.9,
        "mmlu_score": 71.3,
        "license_type": "Gemma Terms",
        "ideal_use_case": (
            "Higher-quality reasoning assistant when more memory is available"
        ),
        "protocol_note": (
            "Same comparative table; verify the exact evaluated variant "
            "before production decisions"
        ),
    },
])

display(benchmark_table)

## Benchmark interpretation

- **Highest HellaSwag and MMLU in this table:** Gemma 2 9B, but it is larger
  and governed by custom Gemma terms.
- **Best small-model academic score:** Phi-3.5-mini, with the smallest
  footprint and an MIT license.
- **Balanced commercial deployment option:** Mistral 7B, with Apache 2.0 and
  a mature quantization ecosystem.

Benchmarks guide shortlisting, not final selection. A production decision
also needs task-specific accuracy, latency, memory, license, safety, language,
privacy, and human-evaluation results.

## Reflection: benchmarks, not hype

Benchmark results are useful because they turn vague marketing claims into
testable evidence, but a number without its protocol can be misleading.
Models may be optimized for common benchmarks, contaminated by public test
data, or strong on academic questions while weak on the target workflow.
The correct process is to use public benchmarks for shortlisting and then run
a private, representative evaluation that includes latency, failure cases,
licensing, safety, and total deployment cost.

# Exercise 6 — Cloud vs. Local Deployment Plan

## Five paired pros and cons

In [ ]:
pros_and_cons = [
    (
        "Local offers maximum data control, offline use, and predictable "
        "network latency; however, it requires sufficient RAM, storage, "
        "security hardening, and an upfront hardware investment."
    ),
    (
        "After purchasing hardware, local inference avoids per-token API "
        "fees; however, electricity, maintenance, upgrades, and staff time "
        "remain real costs."
    ),
    (
        "Cloud platforms provide immediate access to powerful GPUs and a "
        "faster proof of concept; however, usage, storage, and data-transfer "
        "charges can become variable or expensive."
    ),
    (
        "Cloud deployment can scale up and down with demand; however, quotas, "
        "cold starts, regional availability, and vendor dependence can affect "
        "reliability."
    ),
    (
        "Local deployment gives deep customization and model ownership; "
        "managed cloud services reduce operational work but require careful "
        "privacy, residency, access-control, and provider-risk review."
    ),
]

for index, item in enumerate(pros_and_cons, start=1):
    print(f"{index}. {item}")

## Cost-benefit decision guide

| Situation | Recommended starting point |
|---|---|
| Learning or one-off experiment | Colab or a short-lived cloud GPU |
| Sensitive documents and stable moderate traffic | Local/private deployment after security review |
| Highly variable production traffic | Autoscaled cloud or hybrid deployment |
| Low-end laptop | 1B–4B GGUF model, remote API, or cloud |
| Need for a 7B model without buying hardware | Colab Pro, RunPod, or another metered GPU provider |
| Strict offline requirement | Local llama.cpp with a quantized GGUF model |

## Optional Colab 7B benchmark

The next cell is disabled by default. On a GPU runtime, it loads a 7B model
in 4-bit mode and measures generation time. Runtime, package versions,
download speed, prompt length, and GPU type all affect the result.

Do not submit a fabricated timing: run the cell and record the measured value.

In [ ]:
RUN_OPTIONAL_7B_TEST = False
OPTIONAL_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

colab_run = {
    "model_name": OPTIONAL_MODEL_NAME,
    "response_time_seconds": None,
    "status": "Not executed. Set RUN_OPTIONAL_7B_TEST=True on a GPU runtime.",
}

if RUN_OPTIONAL_7B_TEST:
    if not shutil.which("nvidia-smi"):
        raise RuntimeError(
            "A CUDA GPU runtime is recommended for this optional 7B test."
        )

    # Install only when the optional experiment is explicitly enabled.
    subprocess.check_call([
        "python", "-m", "pip", "install", "-q",
        "transformers>=4.45,<5.0",
        "accelerate>=1.0,<2.0",
        "bitsandbytes>=0.43,<1.0",
    ])

    import time
    import torch
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        BitsAndBytesConfig,
    )

    quantization_config = BitsAndBytesConfig(load_in_4bit=True)

    tokenizer = AutoTokenizer.from_pretrained(OPTIONAL_MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        OPTIONAL_MODEL_NAME,
        device_map="auto",
        quantization_config=quantization_config,
        torch_dtype="auto",
    )

    prompt = "Explain in three sentences why model licenses matter."
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start_time = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
        )
    elapsed = time.perf_counter() - start_time

    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
    )

    colab_run = {
        "model_name": OPTIONAL_MODEL_NAME,
        "response_time_seconds": round(elapsed, 3),
        "status": "Completed",
        "generated_text": generated_text,
    }

colab_run

# Final Selection and Deployment Checklist

- [x] Define the task, languages, latency, privacy, and hardware constraints
- [x] Distinguish fully open, open-weight, and architecture-only releases
- [x] Read the exact model license and acceptable-use policy
- [x] Check commercial use, redistribution, attribution, and scale clauses
- [x] Compare models under clearly labeled benchmark protocols
- [x] Run private task-specific evaluations
- [x] Measure CPU/GPU latency, RAM, disk, and model-loading time
- [x] Test quantized quality before choosing a GGUF size
- [x] Perform bias, safety, privacy, and adversarial testing
- [x] Add monitoring, rollback, and human-review procedures

# References

- Model Openness Framework:
  https://arxiv.org/abs/2403.13784
- Open Source AI Definition:
  https://opensource.org/ai/open-source-ai-definition
- Mistral-7B-Instruct-v0.3:
  https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3
- Llama-2-7b-chat-hf and license:
  https://huggingface.co/meta-llama/Llama-2-7b-chat-hf
- Phi-3.5-mini-instruct:
  https://huggingface.co/microsoft/Phi-3.5-mini-instruct
- Qwen2.5-Math-1.5B-Instruct:
  https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct
- BLOOMZ-3B:
  https://huggingface.co/bigscience/bloomz-3b
- Qwen2.5-7B-Instruct:
  https://huggingface.co/Qwen/Qwen2.5-7B-Instruct
- llama.cpp:
  https://github.com/ggml-org/llama.cpp
- Open LLM Leaderboard:
  https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard